# Progressive TESSERA intercropping separability

This notebook tests a direct question: **can held-out physical fields labelled as intercropped be distinguished from both corresponding monocrops using progressive TESSERA embeddings?**

It uses every scoreable canonical field and the same exact 10 m pixels through `w1`-`w4`. The primary evidence is field-level nested cross-validation for two three-class families. Pixel predictions are generated with the same held-out-field folds and are used only to explain where the signal appears; pixels are never counted as independent validation samples.

`w1`, `w1+w2`, and `w1+w2+w3` are the primary progressive tests. `w4` is visibly marked as an out-of-contract sensitivity analysis. The analysis tests embedding separability, not biomass, planted-area fraction, plant count, or yield.

Run all cells. Every PDF-ready figure is displayed and saved at 200 dpi beneath the printed export directory.


In [ ]:
from collections import OrderedDict
from functools import reduce
import hashlib
import json
import os
from pathlib import Path
import tempfile

from IPython import get_ipython
from IPython.display import display

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pyproj import Transformer
from shapely import wkt as shapely_wkt
from shapely.ops import transform as shapely_transform
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


OUTPUT_DIR = Path(
    os.environ.get(
        "TESSERA_OUTPUT_DIR",
        "/mnt/noobjam/harvard_tessera_incremental_v2",
    )
).expanduser().resolve()
EXPORT_BASE = Path(
    os.environ.get(
        "TESSERA_SEPARABILITY_EXPORT_DIR",
        str(OUTPUT_DIR / "analysis" / "intercropping_temporal_separability_v1"),
    )
).expanduser().resolve()
WINDOWS = ("w1", "w2", "w3", "w4")
PRIMARY_WINDOWS = ("w1", "w2", "w3")
STAGES = OrderedDict(
    (
        ("w1", ("w1",)),
        ("w1+w2", ("w1", "w2")),
        ("w1+w2+w3", ("w1", "w2", "w3")),
        ("w1+w2+w3+w4", ("w1", "w2", "w3", "w4")),
    )
)
PRIMARY_STAGE = "w1+w2+w3"
LABELS = (
    "Maize",
    "Bean and Maize",
    "Irish Potato",
    "Bean",
    "Rice",
    "Irish Potato and Maize",
)
TASKS = OrderedDict(
    (
        (
            "bean_maize",
            {
                "title": "Bean - Maize family",
                "classes": ("Bean", "Maize", "Bean and Maize"),
                "mixture": "Bean and Maize",
            },
        ),
        (
            "potato_maize",
            {
                "title": "Irish Potato - Maize family",
                "classes": (
                    "Irish Potato",
                    "Maize",
                    "Irish Potato and Maize",
                ),
                "mixture": "Irish Potato and Maize",
            },
        ),
    )
)
CLASS_COLORS = {
    "Maize": "#D99B16",
    "Bean": "#25855A",
    "Irish Potato": "#7E57A6",
    "Bean and Maize": "#2A9D8F",
    "Irish Potato and Maize": "#B565A7",
    "Rice": "#3A78B4",
}
RANDOM_SEED = 20260709
MAX_OUTER_FOLDS = 5
INNER_FOLDS = 3
CANDIDATE_C = (0.01, 0.1, 1.0)
BOOTSTRAP_REPLICATES = 500
PIXEL_SIZE_M = 10.0

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "font.size": 10.5,
        "axes.titlesize": 12,
        "axes.labelsize": 10.5,
        "legend.fontsize": 8.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def _json(path):
    value = json.loads(Path(path).read_text())
    if not isinstance(value, dict):
        raise ValueError(f"expected JSON object in {path}")
    return value


def _canonical_sha(value):
    encoded = json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode()
    return hashlib.sha256(encoded).hexdigest()


def _task_key(row):
    return _canonical_sha(
        {
            "epsg": int(row.utm_epsg),
            "work_x": int(row.work_x_index),
            "work_y": int(row.work_y_index),
            "chunk_x": int(row.chunk_x_index),
            "chunk_y": int(row.chunk_y_index),
        }
    )[:24]


def _row_id(run_fingerprint, field_uid, pixel_id, window_id):
    text = f"{run_fingerprint}:{field_uid}:{pixel_id}:{window_id}"
    return hashlib.sha256(text.encode()).hexdigest()


def _finite_vector(value):
    if value is None:
        return None
    vector = np.asarray(value, dtype=np.float32)
    if vector.shape != (128,) or not np.isfinite(vector).all():
        return None
    norm = float(np.linalg.norm(vector))
    if norm <= 0:
        return None
    return (vector / norm).astype(np.float32)


def _atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    handle = tempfile.NamedTemporaryFile(
        prefix=f".{path.stem}.",
        suffix=".json.part",
        dir=path.parent,
        delete=False,
    )
    temporary = Path(handle.name)
    handle.close()
    try:
        temporary.write_text(json.dumps(value, indent=2, sort_keys=True, default=str) + "\n")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def _atomic_parquet(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".part")
    frame.to_parquet(temporary, index=False)
    os.replace(temporary, path)


def _save_figure(figure, export_root, filename):
    path = Path(export_root) / "figures" / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.stem}.png.part")
    figure.savefig(temporary, format="png", dpi=200, bbox_inches="tight")
    os.replace(temporary, path)
    return path


def load_longitudinal_embeddings(output_dir):
    output_dir = Path(output_dir)
    required = ("run.json", "fields.parquet", "field_pixels.parquet")
    missing = [name for name in required if not (output_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(f"missing pipeline artifacts: {missing}")

    run = _json(output_dir / "run.json")
    run_fingerprint = str(run["run_fingerprint"])
    fields = pd.read_parquet(
        output_dir / "fields.parquet",
        columns=[
            "field_uid",
            "id",
            "landcover",
            "wkt",
            "utm_epsg",
            "pixel_count",
            "geometry_sha256",
        ],
    )
    membership_columns = [
        "field_uid",
        "landcover",
        "pixel_id",
        "utm_epsg",
        "pixel_x_index",
        "pixel_y_index",
        "pixel_longitude",
        "pixel_latitude",
        "work_x_index",
        "work_y_index",
        "chunk_x_index",
        "chunk_y_index",
    ]
    memberships = pd.read_parquet(
        output_dir / "field_pixels.parquet",
        columns=membership_columns,
    )
    fields["field_uid"] = fields["field_uid"].astype(str)
    fields["landcover"] = fields["landcover"].astype(str).str.strip()
    memberships["field_uid"] = memberships["field_uid"].astype(str)
    memberships["pixel_id"] = memberships["pixel_id"].astype(str)
    memberships["landcover"] = memberships["landcover"].astype(str).str.strip()

    fields["geometry_label_count"] = fields.groupby("geometry_sha256")[
        "landcover"
    ].transform("nunique")
    fields["analysis_unit_uid"] = fields.groupby(
        ["geometry_sha256", "landcover"],
        sort=False,
    )["field_uid"].transform("min")
    fields["source_replica_count"] = fields.groupby(
        ["geometry_sha256", "landcover"],
        sort=False,
    )["field_uid"].transform("size")
    units = fields[
        fields["landcover"].isin(LABELS)
        & fields["geometry_label_count"].eq(1)
        & fields["field_uid"].eq(fields["analysis_unit_uid"])
    ].copy()
    if units["field_uid"].duplicated().any():
        raise RuntimeError("canonical field identifiers are not unique")
    canonical_ids = set(units["field_uid"])
    canonical_memberships = memberships[
        memberships["field_uid"].isin(canonical_ids)
    ].copy()
    if canonical_memberships.duplicated(["field_uid", "pixel_id"]).any():
        raise RuntimeError("canonical field memberships are duplicated")

    task_columns = [
        "utm_epsg",
        "work_x_index",
        "work_y_index",
        "chunk_x_index",
        "chunk_y_index",
    ]
    task_table = canonical_memberships[task_columns].drop_duplicates().copy()
    task_table["task_key"] = [
        _task_key(row) for row in task_table.itertuples(index=False)
    ]
    canonical_memberships = canonical_memberships.merge(
        task_table,
        on=task_columns,
        how="left",
        validate="many_to_one",
    )
    expected_counts = canonical_memberships.groupby("field_uid").size()
    used_files = set()
    loaded_by_window = {}

    for window in WINDOWS:
        window_parts = []
        for task_key, rows in canonical_memberships.groupby("task_key", sort=True):
            path = output_dir / "embeddings" / f"window_id={window}" / f"{task_key}.parquet"
            if not path.is_file():
                continue
            row_ids = [
                _row_id(run_fingerprint, field_uid, pixel_id, window)
                for field_uid, pixel_id in rows[
                    ["field_uid", "pixel_id"]
                ].itertuples(index=False, name=None)
            ]
            table = pq.read_table(
                path,
                columns=["row_id", "field_uid", "pixel_id", "outcome", "embedding"],
                filters=[("row_id", "in", row_ids)],
                partitioning=None,
            ).to_pandas()
            if not table.empty:
                window_parts.append(table)
                used_files.add(path)
        loaded = (
            pd.concat(window_parts, ignore_index=True)
            if window_parts
            else pd.DataFrame(
                columns=["row_id", "field_uid", "pixel_id", "outcome", "embedding"]
            )
        )
        loaded["field_uid"] = loaded["field_uid"].astype(str)
        loaded["pixel_id"] = loaded["pixel_id"].astype(str)
        if loaded.duplicated(["field_uid", "pixel_id"]).any():
            raise RuntimeError(f"duplicate field/pixel rows in {window}")
        loaded["_vector"] = loaded["embedding"].map(_finite_vector)
        loaded_by_window[window] = loaded.drop(columns="embedding")

    published_by_window = {}
    for window, loaded in loaded_by_window.items():
        counts = loaded.groupby("field_uid").size().reindex(expected_counts.index, fill_value=0)
        published_by_window[window] = set(counts[counts.eq(expected_counts)].index)
    common_published = reduce(set.intersection, published_by_window.values())

    pixel_field_count = canonical_memberships.groupby("pixel_id")["field_uid"].nunique()
    private_pixels = set(pixel_field_count[pixel_field_count.eq(1)].index)
    model_memberships = canonical_memberships[
        canonical_memberships["field_uid"].isin(common_published)
        & canonical_memberships["pixel_id"].isin(private_pixels)
    ].copy()
    allowed_pairs = set(
        model_memberships[["field_uid", "pixel_id"]].itertuples(index=False, name=None)
    )
    complete_sets = []
    for window, loaded in loaded_by_window.items():
        complete = loaded[
            loaded["outcome"].eq("complete") & loaded["_vector"].notna()
        ]
        complete_sets.append(
            set(complete[["field_uid", "pixel_id"]].itertuples(index=False, name=None))
            & allowed_pairs
        )
    common_pairs = reduce(set.intersection, complete_sets)
    scoreable_fields = {field_uid for field_uid, _ in common_pairs}
    if not scoreable_fields:
        raise RuntimeError("no complete private field pixels are shared by all windows")

    model_memberships = model_memberships[
        model_memberships["field_uid"].isin(scoreable_fields)
    ].drop(columns="task_key")
    timeline_parts = []
    for window, loaded in loaded_by_window.items():
        selected = loaded[
            loaded[["field_uid", "pixel_id"]]
            .apply(tuple, axis=1)
            .isin(common_pairs)
        ].drop(columns=["row_id", "outcome"])
        selected = selected.merge(
            model_memberships,
            on=["field_uid", "pixel_id"],
            how="left",
            validate="one_to_one",
        )
        selected["window_id"] = window
        timeline_parts.append(selected)
    timeline = pd.concat(timeline_parts, ignore_index=True)
    if timeline.duplicated(["field_uid", "pixel_id", "window_id"]).any():
        raise RuntimeError("longitudinal timeline contains duplicate rows")
    pair_counts = timeline.groupby(["field_uid", "pixel_id"])["window_id"].nunique()
    if not pair_counts.eq(len(WINDOWS)).all():
        raise RuntimeError("the exact pixel cohort is not present in every window")

    units = units[units["field_uid"].isin(scoreable_fields)].copy()
    field_labels = units.set_index("field_uid")["landcover"]
    timeline["landcover"] = timeline["field_uid"].map(field_labels)
    if timeline["landcover"].isna().any():
        raise RuntimeError("a timeline field lost its label")

    specs = {
        str(spec["window_id"]): dict(spec)
        for spec in run.get("config", {}).get("windows", [])
        if isinstance(spec, dict) and "window_id" in spec
    }
    window_records = []
    for window in WINDOWS:
        spec = specs.get(window, {"window_id": window})
        start = pd.Timestamp(str(spec.get("start", "2024-09-01")))
        end = pd.Timestamp(str(spec.get("end_exclusive", "2026-01-01")))
        duration = int((end - start).days)
        window_records.append(
            {
                "window_id": window,
                "start": start.date().isoformat(),
                "end_exclusive": end.date().isoformat(),
                "duration_days": duration,
                "contract_status": "in contract" if duration <= 366 else "OOD sensitivity",
            }
        )
    window_table = pd.DataFrame(window_records)

    file_records = [
        {
            "path": str(path.relative_to(output_dir)),
            "bytes": path.stat().st_size,
            "mtime_ns": path.stat().st_mtime_ns,
        }
        for path in sorted(used_files)
    ]
    snapshot_id = _canonical_sha(
        {
            "run_fingerprint": run_fingerprint,
            "files": file_records,
            "common_pairs": len(common_pairs),
        }
    )[:16]
    diagnostics = {
        "run_fingerprint": run_fingerprint,
        "snapshot_id": snapshot_id,
        "pipeline_complete_at_snapshot": (output_dir / "COMPLETED.json").is_file(),
        "source_field_rows": int(len(fields)),
        "canonical_supported_units": int(len(fields[
            fields["landcover"].isin(LABELS)
            & fields["geometry_label_count"].eq(1)
            & fields["field_uid"].eq(fields["analysis_unit_uid"])
        ])),
        "scoreable_physical_fields": int(len(units)),
        "scoreable_physical_pixels": int(len(common_pairs)),
        "shared_physical_pixels_removed": int(len(pixel_field_count) - len(private_pixels)),
    }
    return {
        "run": run,
        "fields": units.reset_index(drop=True),
        "timeline": timeline.reset_index(drop=True),
        "window_table": window_table,
        "diagnostics": diagnostics,
        "snapshot_id": snapshot_id,
    }


def build_field_features(timeline, fields):
    field_ids = sorted(fields["field_uid"].astype(str))
    field_table = fields.set_index("field_uid").loc[field_ids].reset_index()
    summaries = []
    mean_by_window = {}
    std_by_window = {}
    for window in WINDOWS:
        rows = timeline[timeline["window_id"].eq(window)]
        means = []
        standard_deviations = []
        for field_uid in field_ids:
            vectors = np.stack(
                rows.loc[rows["field_uid"].eq(field_uid), "_vector"].to_numpy()
            ).astype(np.float64)
            mean = vectors.mean(axis=0)
            standard_deviation = vectors.std(axis=0)
            means.append(mean)
            standard_deviations.append(standard_deviation)
            summaries.append(
                {
                    "field_uid": field_uid,
                    "window_id": window,
                    "landcover": str(field_table.loc[
                        field_table["field_uid"].eq(field_uid), "landcover"
                    ].iloc[0]),
                    "pixel_count": int(len(vectors)),
                    "dispersion": float(
                        np.linalg.norm(vectors - mean[None, :], axis=1).mean()
                    ),
                }
            )
        mean_by_window[window] = np.vstack(means)
        std_by_window[window] = np.vstack(standard_deviations)
    features = {
        stage: np.hstack(
            [
                component
                for window in stage_windows
                for component in (mean_by_window[window], std_by_window[window])
            ]
        )
        for stage, stage_windows in STAGES.items()
    }
    return {
        "field_table": field_table,
        "features": features,
        "mean_by_window": mean_by_window,
        "summary": pd.DataFrame(summaries),
    }


def _classifier(c_value=0.1):
    return Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    C=float(c_value),
                    class_weight="balanced",
                    max_iter=3000,
                    solver="lbfgs",
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )


def _metric_values(y_true, y_pred, mixture_probability, classes, mixture):
    binary = np.asarray(y_true) == mixture
    return {
        "macro_f1": float(
            f1_score(y_true, y_pred, labels=list(classes), average="macro", zero_division=0)
        ),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "mixture_recall": float(
            recall_score(binary, np.asarray(y_pred) == mixture, zero_division=0)
        ),
        "mixture_auroc": float(roc_auc_score(binary, mixture_probability)),
        "mixture_average_precision": float(
            average_precision_score(binary, mixture_probability)
        ),
    }


def _bootstrap_intervals(
    y_true,
    y_pred,
    mixture_probability,
    classes,
    mixture,
    seed,
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mixture_probability = np.asarray(mixture_probability, dtype=np.float64)
    rng = np.random.default_rng(seed)
    values = {name: [] for name in _metric_values(
        y_true, y_pred, mixture_probability, classes, mixture
    )}
    class_positions = {
        label: np.flatnonzero(y_true == label) for label in classes
    }
    for _ in range(BOOTSTRAP_REPLICATES):
        sample = np.concatenate(
            [
                rng.choice(positions, size=len(positions), replace=True)
                for positions in class_positions.values()
            ]
        )
        metrics = _metric_values(
            y_true[sample],
            y_pred[sample],
            mixture_probability[sample],
            classes,
            mixture,
        )
        for name, value in metrics.items():
            values[name].append(value)
    return {
        name: tuple(float(value) for value in np.quantile(samples, [0.025, 0.975]))
        for name, samples in values.items()
    }


def run_field_models(field_bundle):
    field_table = field_bundle["field_table"]
    all_labels = field_table["landcover"].astype(str).to_numpy()
    performance_records = []
    prediction_parts = []
    fold_records = []
    for task_offset, (task_key, task) in enumerate(TASKS.items()):
        selected = field_table["landcover"].isin(task["classes"]).to_numpy()
        positions = np.flatnonzero(selected)
        y = all_labels[selected]
        counts = pd.Series(y).value_counts().reindex(task["classes"])
        fold_count = int(min(MAX_OUTER_FOLDS, counts.min()))
        if fold_count < 2:
            raise RuntimeError(f"{task_key} needs at least two physical fields per class")
        outer = StratifiedKFold(
            n_splits=fold_count,
            shuffle=True,
            random_state=RANDOM_SEED + task_offset,
        )
        splits = list(outer.split(np.zeros(len(y)), y))
        for stage_offset, (stage, _) in enumerate(STAGES.items()):
            x = field_bundle["features"][stage][selected]
            oof_probability = np.full((len(y), len(task["classes"])), np.nan)
            oof_prediction = np.empty(len(y), dtype=object)
            oof_fold = np.full(len(y), -1, dtype=np.int64)
            chosen_c = []
            for fold, (train, test) in enumerate(splits):
                train_counts = pd.Series(y[train]).value_counts()
                inner_count = int(min(INNER_FOLDS, train_counts.min()))
                if inner_count < 2:
                    raise RuntimeError(f"{task_key}/{stage} lacks inner-fold support")
                inner = StratifiedKFold(
                    n_splits=inner_count,
                    shuffle=True,
                    random_state=RANDOM_SEED + 100 * task_offset + fold,
                )
                search = GridSearchCV(
                    _classifier(),
                    {"classifier__C": CANDIDATE_C},
                    scoring="f1_macro",
                    cv=inner,
                    refit=True,
                    error_score="raise",
                    n_jobs=1,
                )
                search.fit(x[train], y[train])
                model = search.best_estimator_
                raw_probability = model.predict_proba(x[test])
                class_index = {label: index for index, label in enumerate(model.classes_)}
                aligned = np.column_stack(
                    [raw_probability[:, class_index[label]] for label in task["classes"]]
                )
                oof_probability[test] = aligned
                oof_prediction[test] = np.asarray(task["classes"])[
                    np.argmax(aligned, axis=1)
                ]
                oof_fold[test] = fold
                best_c = float(search.best_params_["classifier__C"])
                chosen_c.append(best_c)
                fold_records.append(
                    {
                        "task_key": task_key,
                        "stage": stage,
                        "fold": fold,
                        "best_c": best_c,
                        "train_fields": tuple(
                            field_table.loc[positions[train], "field_uid"].astype(str)
                        ),
                        "test_fields": tuple(
                            field_table.loc[positions[test], "field_uid"].astype(str)
                        ),
                    }
                )
            if not np.isfinite(oof_probability).all() or (oof_fold < 0).any():
                raise RuntimeError(f"{task_key}/{stage} did not receive complete OOF predictions")
            mixture_index = task["classes"].index(task["mixture"])
            mixture_probability = oof_probability[:, mixture_index]
            metrics = _metric_values(
                y,
                oof_prediction,
                mixture_probability,
                task["classes"],
                task["mixture"],
            )
            intervals = _bootstrap_intervals(
                y,
                oof_prediction,
                mixture_probability,
                task["classes"],
                task["mixture"],
                RANDOM_SEED + 1000 * task_offset + stage_offset,
            )
            record = {
                "task_key": task_key,
                "task_title": task["title"],
                "stage": stage,
                "contract_status": (
                    "primary" if stage != "w1+w2+w3+w4" else "OOD sensitivity"
                ),
                "fold_count": fold_count,
                "fields": len(y),
                "mixture_fields": int(np.sum(y == task["mixture"])),
                "median_selected_c": float(np.median(chosen_c)),
            }
            for name, value in metrics.items():
                low, high = intervals[name]
                record[name] = value
                record[name + "_ci_low"] = low
                record[name + "_ci_high"] = high
            performance_records.append(record)
            predictions = pd.DataFrame(
                {
                    "field_uid": field_table.loc[positions, "field_uid"].astype(str).to_numpy(),
                    "landcover": y,
                    "task_key": task_key,
                    "stage": stage,
                    "fold": oof_fold,
                    "predicted_label": oof_prediction,
                    "mixture_probability": mixture_probability,
                }
            )
            for class_position, label in enumerate(task["classes"]):
                slug = label.lower().replace(" ", "_")
                predictions["prob_" + slug] = oof_probability[:, class_position]
            prediction_parts.append(predictions)
    return {
        "performance": pd.DataFrame(performance_records),
        "predictions": pd.concat(prediction_parts, ignore_index=True),
        "fold_plans": pd.DataFrame(fold_records),
    }


def _pixel_stage_matrix(timeline, task_classes, stage_windows):
    rows = timeline[timeline["landcover"].isin(task_classes)].copy()
    first = rows[rows["window_id"].eq(stage_windows[0])][
        [
            "field_uid",
            "pixel_id",
            "landcover",
            "utm_epsg",
            "pixel_x_index",
            "pixel_y_index",
        ]
    ].sort_values(["field_uid", "pixel_id"], kind="stable")
    key_index = pd.MultiIndex.from_frame(first[["field_uid", "pixel_id"]])
    matrices = []
    for window in stage_windows:
        window_rows = rows[rows["window_id"].eq(window)].set_index(
            ["field_uid", "pixel_id"]
        )
        window_rows = window_rows.reindex(key_index)
        if window_rows["_vector"].isna().any():
            raise RuntimeError(f"pixel feature alignment failed in {window}")
        matrices.append(np.stack(window_rows["_vector"].to_numpy()))
    return first.reset_index(drop=True), np.hstack(matrices)


def run_pixel_models(timeline, field_results):
    prediction_parts = []
    stages = {key: STAGES[key] for key in ("w1", "w1+w2", PRIMARY_STAGE)}
    for task_key, task in TASKS.items():
        for stage, stage_windows in stages.items():
            keys, x = _pixel_stage_matrix(timeline, task["classes"], stage_windows)
            y = keys["landcover"].astype(str).to_numpy()
            predictions = np.full((len(keys), len(task["classes"])), np.nan)
            plans = field_results["fold_plans"][
                field_results["fold_plans"]["task_key"].eq(task_key)
                & field_results["fold_plans"]["stage"].eq(stage)
            ]
            for plan in plans.itertuples(index=False):
                train_mask = keys["field_uid"].astype(str).isin(plan.train_fields).to_numpy()
                test_mask = keys["field_uid"].astype(str).isin(plan.test_fields).to_numpy()
                train_keys = keys.loc[train_mask]
                field_pixel_count = train_keys.groupby("field_uid").size()
                class_field_count = train_keys.groupby("landcover")["field_uid"].nunique()
                weights = np.asarray(
                    [
                        1.0
                        / (
                            class_field_count[label]
                            * field_pixel_count[field_uid]
                        )
                        for field_uid, label in train_keys[
                            ["field_uid", "landcover"]
                        ].itertuples(index=False, name=None)
                    ],
                    dtype=np.float64,
                )
                weights *= len(weights) / weights.sum()
                model = _classifier(plan.best_c)
                model.fit(
                    x[train_mask],
                    y[train_mask],
                    classifier__sample_weight=weights,
                )
                raw = model.predict_proba(x[test_mask])
                class_index = {label: index for index, label in enumerate(model.classes_)}
                predictions[test_mask] = np.column_stack(
                    [raw[:, class_index[label]] for label in task["classes"]]
                )
            if not np.isfinite(predictions).all():
                raise RuntimeError(f"pixel OOF predictions are incomplete for {task_key}/{stage}")
            mixture_index = task["classes"].index(task["mixture"])
            part = keys.copy()
            part["task_key"] = task_key
            part["stage"] = stage
            part["mixture_probability"] = predictions[:, mixture_index]
            prediction_parts.append(part)
    return pd.concat(prediction_parts, ignore_index=True)


def figure_cohort(field_bundle, window_table, diagnostics):
    counts = field_bundle["field_table"]["landcover"].value_counts().reindex(LABELS)
    figure, axes = plt.subplots(1, 2, figsize=(13, 5.2))
    colors = [CLASS_COLORS[label] for label in counts.index]
    axes[0].barh(counts.index, counts.to_numpy(), color=colors)
    for position, count in enumerate(counts):
        axes[0].text(count, position, f" {count:,}", va="center")
    axes[0].set_title("Physical fields in the frozen four-window cohort")
    axes[0].set_xlabel("fields")
    axes[0].grid(axis="x", alpha=0.18)

    durations = window_table["duration_days"].to_numpy()
    bar_colors = ["#4C78A8" if status == "in contract" else "#9E9E9E" for status in window_table["contract_status"]]
    axes[1].bar(window_table["window_id"], durations, color=bar_colors)
    for position, row in enumerate(window_table.itertuples(index=False)):
        axes[1].text(position, row.duration_days + 8, f"{row.duration_days} d", ha="center")
    axes[1].set_ylim(0, max(durations) * 1.15)
    axes[1].set_ylabel("cumulative days from 2024-09-01")
    axes[1].set_title("Progressive TESSERA windows")
    axes[1].text(
        0.02,
        0.95,
        f"{diagnostics['scoreable_physical_pixels']:,} exact 10 m pixels\n"
        f"w4 is sensitivity only",
        transform=axes[1].transAxes,
        va="top",
        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "#CCCCCC"},
    )
    axes[1].grid(axis="y", alpha=0.18)
    figure.suptitle("Intercropping temporal-separability cohort", fontsize=15)
    figure.tight_layout(rect=(0, 0, 1, 0.95))
    return figure


def figure_performance(performance):
    figure, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
    metrics = (
        ("macro_f1", "Macro-F1", "#3B76AF"),
        ("mixture_recall", "Intercrop recall", "#2A9D8F"),
        ("mixture_auroc", "Intercrop AUROC", "#D97706"),
    )
    stages = list(STAGES)
    x = np.arange(len(stages))
    for axis, (task_key, task) in zip(axes, TASKS.items(), strict=True):
        rows = performance[performance["task_key"].eq(task_key)].set_index("stage").loc[stages]
        for metric, label, color in metrics:
            values = rows[metric].to_numpy()
            low = rows[metric + "_ci_low"].to_numpy()
            high = rows[metric + "_ci_high"].to_numpy()
            axis.errorbar(
                x,
                values,
                yerr=np.vstack((values - low, high - values)),
                marker="o",
                capsize=3,
                linewidth=2,
                color=color,
                label=label,
            )
        axis.axvspan(2.5, 3.5, color="#EEEEEE", zorder=-2)
        axis.text(3, 0.04, "w4 OOD", ha="center", color="#555555")
        axis.set_xticks(x, ("w1", "w1+w2", "w1+w2+w3", "+w4"), rotation=15)
        axis.set_ylim(0, 1.03)
        axis.set_title(task["title"])
        axis.grid(alpha=0.18)
    axes[0].set_ylabel("held-out field score")
    axes[1].legend(loc="lower left")
    figure.suptitle("Does progressive TESSERA separate the intercrop from both parents?", fontsize=15)
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    return figure


def figure_confusions(predictions):
    figure, axes = plt.subplots(1, 2, figsize=(13, 5.4))
    for axis, (task_key, task) in zip(axes, TASKS.items(), strict=True):
        rows = predictions[
            predictions["task_key"].eq(task_key)
            & predictions["stage"].eq(PRIMARY_STAGE)
        ]
        matrix = confusion_matrix(
            rows["landcover"],
            rows["predicted_label"],
            labels=list(task["classes"]),
            normalize="true",
        )
        image = axis.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
        for row in range(len(task["classes"])):
            for column in range(len(task["classes"])):
                axis.text(
                    column,
                    row,
                    f"{100 * matrix[row, column]:.0f}%",
                    ha="center",
                    va="center",
                    color="white" if matrix[row, column] > 0.55 else "black",
                    fontsize=11,
                )
        short = [label.replace(" and ", "+") for label in task["classes"]]
        axis.set_xticks(range(len(short)), short, rotation=20, ha="right")
        axis.set_yticks(range(len(short)), short)
        axis.set_xlabel("predicted")
        axis.set_ylabel("true")
        axis.set_title(task["title"])
    figure.colorbar(image, ax=axes, fraction=0.025, pad=0.04, label="within-class rate")
    figure.suptitle("Held-out physical-field confusion after w1+w2+w3", fontsize=15)
    figure.subplots_adjust(top=0.86, bottom=0.20, wspace=0.35)
    return figure


def figure_probability_trajectories(predictions):
    figure, axes = plt.subplots(1, 2, figsize=(14, 5.4), sharey=True)
    stages = list(STAGES)
    x = np.arange(len(stages))
    for axis, (task_key, task) in zip(axes, TASKS.items(), strict=True):
        rows = predictions[predictions["task_key"].eq(task_key)]
        for label in task["classes"]:
            selected = rows[rows["landcover"].eq(label)]
            grouped = selected.groupby("stage")["mixture_probability"]
            median = grouped.median().reindex(stages)
            low = grouped.quantile(0.25).reindex(stages)
            high = grouped.quantile(0.75).reindex(stages)
            color = CLASS_COLORS[label]
            axis.fill_between(x, low, high, color=color, alpha=0.14)
            axis.plot(x, median, marker="o", linewidth=2, color=color, label=label)
        axis.axvspan(2.5, 3.5, color="#EEEEEE", zorder=-2)
        axis.set_xticks(x, ("w1", "w1+w2", "w1+w2+w3", "+w4"), rotation=15)
        axis.set_title(task["title"])
        axis.grid(alpha=0.18)
        axis.legend(fontsize=8)
    axes[0].set_ylabel("held-out probability assigned to the intercrop class")
    axes[0].set_ylim(0, 1)
    figure.suptitle("Intercrop probability through cumulative windows (median and IQR)", fontsize=15)
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    return figure


def figure_field_space(field_bundle):
    figure, axes = plt.subplots(1, 2, figsize=(14, 5.8))
    field_table = field_bundle["field_table"]
    for axis, (task_key, task) in zip(axes, TASKS.items(), strict=True):
        mask = field_table["landcover"].isin(task["classes"]).to_numpy()
        positions = np.flatnonzero(mask)
        stacked = np.vstack(
            [field_bundle["mean_by_window"][window][mask] for window in PRIMARY_WINDOWS]
        )
        standardized = StandardScaler().fit_transform(stacked)
        projected = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(standardized)
        per_window = np.split(projected, len(PRIMARY_WINDOWS))
        w3 = per_window[-1]
        labels = field_table.loc[positions, "landcover"].astype(str).to_numpy()
        for label in task["classes"]:
            selected = labels == label
            axis.scatter(
                w3[selected, 0],
                w3[selected, 1],
                s=22,
                alpha=0.45,
                color=CLASS_COLORS[label],
                label=label,
            )
            centroids = np.vstack(
                [points[selected].mean(axis=0) for points in per_window]
            )
            axis.plot(
                centroids[:, 0],
                centroids[:, 1],
                marker="o",
                linewidth=2.2,
                color=CLASS_COLORS[label],
            )
            for window, point in zip(PRIMARY_WINDOWS, centroids, strict=True):
                axis.text(point[0], point[1], window, fontsize=7, color=CLASS_COLORS[label])
        axis.set_title(task["title"])
        axis.set_xlabel("PCA 1")
        axis.set_ylabel("PCA 2")
        axis.grid(alpha=0.15)
        axis.legend(fontsize=7.5)
    figure.suptitle(
        "Field-mean embedding space: w3 fields and class-centroid trajectories",
        fontsize=15,
    )
    figure.text(
        0.5,
        0.01,
        "PCA is visualization only; all reported classification metrics use the full field feature vectors.",
        ha="center",
        fontsize=9,
    )
    figure.tight_layout(rect=(0, 0.04, 1, 0.94))
    return figure


def figure_heterogeneity(field_summary):
    figure, axes = plt.subplots(1, 2, figsize=(14, 5.4), sharey=True)
    x = np.arange(len(PRIMARY_WINDOWS))
    for axis, (task_key, task) in zip(axes, TASKS.items(), strict=True):
        rows = field_summary[field_summary["landcover"].isin(task["classes"])]
        for label in task["classes"]:
            selected = rows[rows["landcover"].eq(label)]
            grouped = selected.groupby("window_id")["dispersion"]
            median = grouped.median().reindex(PRIMARY_WINDOWS)
            low = grouped.quantile(0.25).reindex(PRIMARY_WINDOWS)
            high = grouped.quantile(0.75).reindex(PRIMARY_WINDOWS)
            color = CLASS_COLORS[label]
            axis.fill_between(x, low, high, color=color, alpha=0.14)
            axis.plot(x, median, marker="o", linewidth=2, color=color, label=label)
        axis.set_xticks(x, PRIMARY_WINDOWS)
        axis.set_title(task["title"])
        axis.grid(alpha=0.18)
        axis.legend(fontsize=8)
    axes[0].set_ylabel("mean pixel distance from its field mean (128-D)")
    figure.suptitle("Within-field embedding heterogeneity (median and IQR)", fontsize=15)
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    return figure


def _iter_polygons(geometry):
    if geometry.geom_type == "Polygon":
        yield geometry
    elif geometry.geom_type in {"MultiPolygon", "GeometryCollection"}:
        for child in geometry.geoms:
            yield from _iter_polygons(child)


def _draw_outline(axis, geometry):
    for polygon in _iter_polygons(geometry):
        x, y = polygon.exterior.xy
        axis.plot(x, y, color="black", linewidth=1.2, zorder=4)


def _representative_mixture_field(task_key, field_results, field_summary):
    task = TASKS[task_key]
    rows = field_results["predictions"][
        field_results["predictions"]["task_key"].eq(task_key)
        & field_results["predictions"]["stage"].eq(PRIMARY_STAGE)
        & field_results["predictions"]["landcover"].eq(task["mixture"])
    ].copy()
    counts = field_summary[field_summary["window_id"].eq("w3")].set_index("field_uid")[
        "pixel_count"
    ]
    rows["pixel_count"] = rows["field_uid"].map(counts)
    eligible = rows[rows["pixel_count"].ge(4)]
    correct = eligible[eligible["predicted_label"].eq(task["mixture"])]
    pool = correct if not correct.empty else eligible
    if pool.empty:
        pool = rows
    target = float(pool["mixture_probability"].median())
    pool = pool.assign(
        _distance=(pool["mixture_probability"] - target).abs()
    ).sort_values(["_distance", "pixel_count", "field_uid"], ascending=[True, False, True])
    selected = pool.iloc[0]
    return {
        "task_key": task_key,
        "field_uid": str(selected["field_uid"]),
        "landcover": task["mixture"],
        "selection_basis": (
            "median-confidence correctly classified held-out intercrop field"
            if not correct.empty
            else "median-confidence held-out intercrop field; none correctly classified"
        ),
        "field_probability_w3": float(selected["mixture_probability"]),
        "pixel_count": int(selected["pixel_count"]),
    }


def figure_pixel_evolution(pixel_predictions, field_results, field_bundle, timeline):
    selections = [
        _representative_mixture_field(task_key, field_results, field_bundle["summary"])
        for task_key in TASKS
    ]
    stages = ("w1", "w1+w2", PRIMARY_STAGE)
    figure, axes = plt.subplots(2, 3, figsize=(15, 9))
    collection = None
    fields = field_bundle["field_table"].set_index("field_uid")
    for row_index, selection in enumerate(selections):
        task_key = selection["task_key"]
        field_uid = selection["field_uid"]
        field_row = fields.loc[field_uid]
        pixel_rows = timeline[
            timeline["field_uid"].eq(field_uid) & timeline["window_id"].eq("w3")
        ][["pixel_id", "utm_epsg", "pixel_x_index", "pixel_y_index"]]
        epsg_values = pixel_rows["utm_epsg"].unique()
        if len(epsg_values) != 1:
            raise RuntimeError(f"field {field_uid} crosses UTM zones")
        epsg = int(epsg_values[0])
        transformer = Transformer.from_crs(4326, epsg, always_xy=True)
        geometry = shapely_transform(
            transformer.transform,
            shapely_wkt.loads(str(field_row["wkt"])),
        )
        left = pixel_rows["pixel_x_index"].to_numpy() * PIXEL_SIZE_M
        bottom = pixel_rows["pixel_y_index"].to_numpy() * PIXEL_SIZE_M
        min_x, min_y, max_x, max_y = geometry.bounds
        min_x = min(min_x, float(left.min()))
        min_y = min(min_y, float(bottom.min()))
        max_x = max(max_x, float((left + PIXEL_SIZE_M).max()))
        max_y = max(max_y, float((bottom + PIXEL_SIZE_M).max()))
        pad = max(PIXEL_SIZE_M, 0.05 * max(max_x - min_x, max_y - min_y))
        for column_index, stage in enumerate(stages):
            axis = axes[row_index, column_index]
            stage_rows = pixel_predictions[
                pixel_predictions["task_key"].eq(task_key)
                & pixel_predictions["stage"].eq(stage)
                & pixel_predictions["field_uid"].eq(field_uid)
            ].copy()
            rectangles = [
                Rectangle(
                    (
                        float(row.pixel_x_index) * PIXEL_SIZE_M,
                        float(row.pixel_y_index) * PIXEL_SIZE_M,
                    ),
                    PIXEL_SIZE_M,
                    PIXEL_SIZE_M,
                )
                for row in stage_rows.itertuples(index=False)
            ]
            collection = PatchCollection(
                rectangles,
                cmap="magma",
                norm=Normalize(0, 1),
                edgecolor="white",
                linewidth=0.35,
            )
            collection.set_array(stage_rows["mixture_probability"].to_numpy())
            axis.add_collection(collection)
            _draw_outline(axis, geometry)
            field_probability = field_results["predictions"].loc[
                field_results["predictions"]["task_key"].eq(task_key)
                & field_results["predictions"]["stage"].eq(stage)
                & field_results["predictions"]["field_uid"].eq(field_uid),
                "mixture_probability",
            ].iloc[0]
            axis.set_title(
                f"{stage}\nfield P={field_probability:.2f}, "
                f"mean pixel P={stage_rows['mixture_probability'].mean():.2f}"
            )
            axis.set_xlim(min_x - pad, max_x + pad)
            axis.set_ylim(min_y - pad, max_y + pad)
            axis.set_aspect("equal")
            axis.set_xticks([])
            axis.set_yticks([])
        axes[row_index, 0].set_ylabel(
            f"{TASKS[task_key]['mixture']}\n{field_uid}",
            fontsize=10,
        )
    if collection is not None:
        colorbar = figure.colorbar(
            collection,
            ax=axes,
            orientation="horizontal",
            fraction=0.035,
            pad=0.07,
        )
        colorbar.set_label("held-out pixel probability of the intercropping class")
    figure.suptitle(
        "Representative fields: where the progressive intercropping signal appears",
        fontsize=15,
    )
    figure.subplots_adjust(top=0.88, bottom=0.13, hspace=0.28, wspace=0.16)
    return figure, pd.DataFrame(selections)


def build_facts(data_bundle, field_bundle, field_results, selections):
    performance_columns = [
        "task_key",
        "task_title",
        "stage",
        "contract_status",
        "fold_count",
        "fields",
        "mixture_fields",
        "macro_f1",
        "macro_f1_ci_low",
        "macro_f1_ci_high",
        "balanced_accuracy",
        "mixture_recall",
        "mixture_recall_ci_low",
        "mixture_recall_ci_high",
        "mixture_auroc",
        "mixture_auroc_ci_low",
        "mixture_auroc_ci_high",
        "mixture_average_precision",
    ]
    counts = (
        field_bundle["field_table"].groupby("landcover", as_index=False)
        .agg(physical_fields=("field_uid", "nunique"))
        .sort_values("physical_fields", ascending=False)
    )
    return {
        "analysis_name": "intercropping_temporal_separability_v1",
        "analysis_question": (
            "Can progressive field-level TESSERA embeddings distinguish each "
            "intercropping class from both corresponding monocrops on held-out fields?"
        ),
        "diagnostics": data_bundle["diagnostics"],
        "windows": json.loads(data_bundle["window_table"].to_json(orient="records")),
        "field_counts": json.loads(counts.to_json(orient="records")),
        "evaluation": {
            "unit": "canonical physical field",
            "outer_validation": "stratified held-out-field folds",
            "inner_selection": "nested macro-F1 selection of L2 regularization",
            "pixel_maps": "held-out-field predictions; pixels are not counted as independent tests",
            "w4_policy": "OOD sensitivity only",
            "geographic_scope": "field generalization inside this snapshot; not a geographic transfer claim",
        },
        "performance": json.loads(
            field_results["performance"][performance_columns].to_json(
                orient="records",
                double_precision=5,
            )
        ),
        "representative_fields": json.loads(selections.to_json(orient="records")),
        "interpretation_boundary": (
            "This analysis tests class separability and spatial-temporal signature, "
            "not biomass, planted-area fraction, plant count, or yield."
        ),
    }


## 1. Freeze the longitudinal physical-field cohort

Exact same-label geometry replicas are represented once, conflicting-label geometries are excluded, shared physical pixels are removed, and only finite embeddings present in every window are retained.


In [ ]:
data_bundle = load_longitudinal_embeddings(OUTPUT_DIR)
field_bundle = build_field_features(data_bundle["timeline"], data_bundle["fields"])
export_root = EXPORT_BASE / data_bundle["snapshot_id"]
export_root.mkdir(parents=True, exist_ok=True)
(export_root / "COMPLETED.json").unlink(missing_ok=True)

print("Input:", OUTPUT_DIR)
print("Frozen export:", export_root)
print(json.dumps(data_bundle["diagnostics"], indent=2))
display(data_bundle["window_table"])
display(
    field_bundle["field_table"]
    .groupby("landcover")
    .agg(physical_fields=("field_uid", "nunique"))
    .sort_values("physical_fields", ascending=False)
)


## 2. Held-out-field progressive classification

For each field and window, the model receives the 128-D mean and 128-D standard deviation across its pixels. Progressive stages concatenate these summaries through the available windows. L2 regularization is selected inside each training fold; the displayed predictions are always out of fold.


In [ ]:
field_results = run_field_models(field_bundle)
pixel_predictions = run_pixel_models(data_bundle["timeline"], field_results)

tables = export_root / "tables"
_atomic_parquet(field_results["performance"], tables / "field_performance.parquet")
_atomic_parquet(field_results["predictions"], tables / "field_oof_predictions.parquet")
_atomic_parquet(pixel_predictions, tables / "pixel_oof_predictions.parquet")
_atomic_parquet(field_bundle["summary"], tables / "field_window_summary.parquet")

display(
    field_results["performance"][
        [
            "task_title",
            "stage",
            "fields",
            "mixture_fields",
            "macro_f1",
            "mixture_recall",
            "mixture_auroc",
            "mixture_average_precision",
        ]
    ].round(3)
)


## Figure 1. Cohort and progressive-window design

Use this figure to establish the denominators and to separate the three valid progressive windows from the `w4` sensitivity window.


In [ ]:
cohort_figure = figure_cohort(
    field_bundle,
    data_bundle["window_table"],
    data_bundle["diagnostics"],
)
cohort_path = _save_figure(cohort_figure, export_root, "01_cohort_and_windows.png")
display(cohort_figure)
plt.close(cohort_figure)
print(cohort_path)


## Figure 2. Progressive held-out-field performance

This is the primary PDF result. It shows whether additional cumulative windows improve three-class separation and specifically the recall and AUROC of the intercropping class. Error bars are stratified field-bootstrap 95% intervals.


In [ ]:
performance_figure = figure_performance(field_results["performance"])
performance_path = _save_figure(
    performance_figure,
    export_root,
    "02_progressive_field_performance.png",
)
display(performance_figure)
plt.close(performance_figure)
print(performance_path)


## Figure 3. What the model confuses after one in-contract year

Rows are true field labels and columns are held-out predictions. Each row sums to 100%, so the minority intercrop class remains visible.


In [ ]:
confusion_figure = figure_confusions(field_results["predictions"])
confusion_path = _save_figure(
    confusion_figure,
    export_root,
    "03_w3_normalized_confusions.png",
)
display(confusion_figure)
plt.close(confusion_figure)
print(confusion_path)


## Figure 4. How intercropping probability emerges through the windows

For each true class, the line is the median held-out probability assigned to the intercrop class and the band is the interquartile range. Useful separation means the intercrop line rises while both parent lines stay low.


In [ ]:
probability_figure = figure_probability_trajectories(
    field_results["predictions"]
)
probability_path = _save_figure(
    probability_figure,
    export_root,
    "04_intercrop_probability_trajectories.png",
)
display(probability_figure)
plt.close(probability_figure)
print(probability_path)


## Figure 5. Field embedding-space context

The points are `w3` field means and the connected markers are class-centroid motion from `w1` to `w3`. This PCA is explanatory only; all reported metrics use the full field feature vectors.


In [ ]:
space_figure = figure_field_space(field_bundle)
space_path = _save_figure(
    space_figure,
    export_root,
    "05_field_embedding_space.png",
)
display(space_figure)
plt.close(space_figure)
print(space_path)


## Figure 6. Within-field spatial heterogeneity

This tests a non-DNA explanation: intercropping may be distinguishable because its pixels form a broader or differently evolving field distribution, even when the field mean is parent-like.


In [ ]:
heterogeneity_figure = figure_heterogeneity(field_bundle["summary"])
heterogeneity_path = _save_figure(
    heterogeneity_figure,
    export_root,
    "06_within_field_heterogeneity.png",
)
display(heterogeneity_figure)
plt.close(heterogeneity_figure)
print(heterogeneity_path)


## Figure 7. Pixel-level explanation inside representative held-out fields

Each row follows one deterministic, median-confidence intercropped field through the primary stages. Colors are out-of-fold pixel probabilities from models that never trained on the displayed field. The field-level metrics remain the proof; this map explains where the learned signal occurs.


In [ ]:
pixel_figure, representative_fields = figure_pixel_evolution(
    pixel_predictions,
    field_results,
    field_bundle,
    data_bundle["timeline"],
)
pixel_path = _save_figure(
    pixel_figure,
    export_root,
    "07_representative_oof_pixel_evolution.png",
)
display(pixel_figure)
plt.close(pixel_figure)
display(representative_fields)
print(pixel_path)


## PDF fact sheet and completion audit

Copy the text block below together with Figures 1-7. The fact sheet contains the exact field-level scores and uncertainty intervals needed to decide which claims the PDF can support.


In [ ]:
facts = build_facts(
    data_bundle,
    field_bundle,
    field_results,
    representative_fields,
)
figure_paths = [
    cohort_path,
    performance_path,
    confusion_path,
    probability_path,
    space_path,
    heterogeneity_path,
    pixel_path,
]
manifest = {
    "analysis_name": facts["analysis_name"],
    "snapshot_id": data_bundle["snapshot_id"],
    "figures": [
        {
            "path": str(path.relative_to(export_root)),
            "bytes": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
        for path in figure_paths
    ],
    "tables": [
        str(path.relative_to(export_root))
        for path in sorted((export_root / "tables").glob("*.parquet"))
    ],
}
_atomic_json(facts, export_root / "PDF_FACTS.json")
_atomic_json(manifest, export_root / "manifest.json")
_atomic_json(
    {
        "analysis_name": facts["analysis_name"],
        "snapshot_id": data_bundle["snapshot_id"],
        "figure_count": len(figure_paths),
        "complete": True,
    },
    export_root / "COMPLETED.json",
)

print("SEPARABILITY_PDF_FACTS_BEGIN")
print(json.dumps(facts, indent=2, allow_nan=False))
print("SEPARABILITY_PDF_FACTS_END")
print("Completed export:", export_root)
